### **Window** Functions

| **Window Function**   | **Usage & Syntax**                          | **Description**                                                                                          |
| --------------------- | ------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| `row_number()`        | `row_number().over(windowSpec)`             | Assigns a unique, sequential number to each row within a partition (starts at 1).                        |
| `rank()`              | `rank().over(windowSpec)`                   | Assigns a rank to each row within a partition. Rows with the same value get the same rank **with gaps**. |
| `dense_rank()`        | `dense_rank().over(windowSpec)`             | Like `rank()`, but **no gaps** in ranking when there are ties.                                           |
| `percent_rank()`      | `percent_rank().over(windowSpec)`           | Computes the relative rank of a row (as a percentile) within the window partition.                       |
| `ntile(n)`            | `ntile(4).over(windowSpec)`                 | Divides partition into `n` equal parts and assigns a bucket number to each row.                          |
| `cume_dist()`         | `cume_dist().over(windowSpec)`              | Calculates the cumulative distribution (proportion of rows ≤ current row).                               |
| `lag()`               | `lag(col("sales"), 1).over(windowSpec)`     | Returns the **previous row’s value** in the same column (based on offset).                               |
| `lag()` with default  | `lag(col("sales"), 1, 0).over(windowSpec)`  | Same as above, but returns `0` if the previous row doesn't exist.                                        |
| `lead()`              | `lead(col("sales"), 1).over(windowSpec)`    | Returns the **next row’s value** in the same column (based on offset).                                   |
| `lead()` with default | `lead(col("sales"), 1, 0).over(windowSpec)` | Returns `0` or given default if the next row doesn't exist.                                              |


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *

In [0]:
spark = SparkSession.builder.appName('SparkByExamples.com').getOrCreate()

simpleData = (("James", "Sales", 3000), \
    ("Michael", "Sales", 4600),  \
    ("Robert", "Sales", 4100),   \
    ("Maria", "Finance", 3000),  \
    ("James", "Sales", 3000),    \
    ("Scott", "Finance", 3300),  \
    ("Jen", "Finance", 3900),    \
    ("Jeff", "Marketing", 3000), \
    ("Kumar", "Marketing", 2000),\
    ("Saif", "Sales", 4100) \
  )
 
columns= ["employee_name", "department", "salary"]
df = spark.createDataFrame(data = simpleData, schema = columns)
df.printSchema()
df.show(truncate=False)

root
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)

+-------------+----------+------+
|employee_name|department|salary|
+-------------+----------+------+
|James        |Sales     |3000  |
|Michael      |Sales     |4600  |
|Robert       |Sales     |4100  |
|Maria        |Finance   |3000  |
|James        |Sales     |3000  |
|Scott        |Finance   |3300  |
|Jen          |Finance   |3900  |
|Jeff         |Marketing |3000  |
|Kumar        |Marketing |2000  |
|Saif         |Sales     |4100  |
+-------------+----------+------+



In [0]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("department").orderBy("salary")
display(windowSpec)
row_number_df = df.withColumn(col("row_number").over(windowSpec).show()
rank_df = df.withColumn("rank").over(windowSpec).show()
dense_rankDf = df.withColumn("dense_rank").over(windowSpec).show()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4001527610379726>:5
      3 windowSpec = Window.partitionBy("department").orderBy("salary")
      4 display(windowSpec)
----> 5 row_number_df = df.withColumn(df["row_number"]).over(windowSpec).show()
      6 rank_df = df.withColumn("rank").over(windowSpec).show()
      7 dense_rankDf = df.withColumn("dense_rank").over(windowSpec).show()

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/dataframe.py:2918, in DataFrame.__getitem__(self, item)
   2876 """Returns the 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

# Define window specification
windowSpec = Window.partitionBy("department").orderBy(col("salary").desc())

# Apply row_number
row_number_df = df.withColumn("row_number", row_number().over(windowSpec))

# Apply rank
rank_df = row_number_df.withColumn("rank", rank().over(windowSpec))

dense_rank_df = rank_df.withColumn("dense_rank", dense_rank().over(windowSpec))
dense_rank_df.display()


employee_name,department,salary,row_number,rank,dense_rank
Jen,Finance,3900,1,1,1
Scott,Finance,3300,2,2,2
Maria,Finance,3000,3,3,3
Jeff,Marketing,3000,1,1,1
Kumar,Marketing,2000,2,2,2
Michael,Sales,4600,1,1,1
Robert,Sales,4100,2,2,2
Saif,Sales,4100,3,2,2
James,Sales,3000,4,4,3
James,Sales,3000,5,4,3


In [0]:
lag_df = df.withColumn("lag", lag(col("salary"), 1).over(windowSpec))
lead_df = lag_df.withColumn("lead", lead(col("salary"), 1).over(windowSpec))
lead_df.display()

employee_name,department,salary,lag,lead
Jen,Finance,3900,null,3300
Scott,Finance,3300,3900,3000
Maria,Finance,3000,3300,null
Jeff,Marketing,3000,null,2000
Kumar,Marketing,2000,3000,null
Michael,Sales,4600,null,4100
Robert,Sales,4100,4600,4100
Saif,Sales,4100,4100,3000
James,Sales,3000,4100,3000
James,Sales,3000,3000,null


In [0]:
aggregated_df = df.withColumn("sum", sum(col("salary")).over(windowSpec)) \
        .withColumn("max", max(col("salary")).over(windowSpec)) \
            .withColumn("min", min(col("salary")).over(windowSpec)) \
                .withColumn("Avg", avg(col("salary")).over(windowSpec)) 

aggregated_df.display()

employee_name,department,salary,sum,max,min,Avg
Jen,Finance,3900,3900,3900,3900,3900.0
Scott,Finance,3300,7200,3900,3300,3600.0
Maria,Finance,3000,10200,3900,3000,3400.0
Jeff,Marketing,3000,3000,3000,3000,3000.0
Kumar,Marketing,2000,5000,3000,2000,2500.0
Michael,Sales,4600,4600,4600,4600,4600.0
Robert,Sales,4100,12800,4600,4100,4266.666666666667
Saif,Sales,4100,12800,4600,4100,4266.666666666667
James,Sales,3000,18800,4600,3000,3760.0
James,Sales,3000,18800,4600,3000,3760.0
